<a href="https://colab.research.google.com/github/vvrgit/NLP-LAB/blob/main/HuggingFaceTutorial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**This installs:**

Transformers (Hugging Face library)
Hugging Face datasets library

In [ ]:
!pip install transformers datasets

# **Import Libraries**

In [ ]:
from transformers import pipeline

# **Sentiment Analysis**

This line automatically:

Downloads model
Loads tokenizer
Initializes pipeline

In [ ]:
classifier = pipeline("sentiment-analysis")

result = classifier("Hugging Face is very easy to use!")
print(result)

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

[{'label': 'POSITIVE', 'score': 0.9988467693328857}]


# **Part 2: Using BERT Model**

**Load Tokenizer & Model**

In [ ]:
from transformers import AutoTokenizer, AutoModel

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
model = AutoModel.from_pretrained("bert-base-uncased")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


**Tokenization**

In [ ]:
text = "Transformers are powerful models"

inputs = tokenizer(text,return_tensors="pt")
print(inputs)

{'input_ids': tensor([[  101, 19081,  2024,  3928,  4275,   102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1]])}


In [ ]:
outputs = model(**inputs)

# CLS token embedding
cls_embedding = outputs.last_hidden_state[:, 0, :]
print(cls_embedding.shape)

torch.Size([1, 768])


# **Text Classification Using Hugging Face With Sample Data**

**Text → Tokenizer → BERT → CLS Embedding → Naive Bayes → Prediction**

In [ ]:
from transformers import AutoTokenizer, AutoModel
import torch
import numpy as np
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import train_test_split

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
model = AutoModel.from_pretrained("bert-base-uncased")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
texts = [
    "I love this product",
    "This is amazing",
    "I am very happy",
    "I hate this",
    "This is bad",
    "Very disappointing"
]

labels = [1, 1, 1, 0, 0, 0]

In [ ]:
def get_embedding(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)

    outputs = model(**inputs)

    # CLS embedding
    cls_embedding = outputs.last_hidden_state[:, 0, :]

    return cls_embedding.detach().numpy()[0]

In [ ]:
X = np.array([get_embedding(t) for t in texts])
y = np.array(labels)

print("Feature shape:", X.shape)

Feature shape: (6, 768)


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

In [ ]:
clf = GaussianNB()
clf.fit(X_train, y_train)

GaussianNB()

In [18]:
y_pred = clf.predict(X_test)

print("Predictions:", y_pred)
print("Actual:", y_test)

Predictions: [1 0]
Actual: [1 0]


In [19]:
test_text = "This product is really good"

test_emb = get_embedding(test_text).reshape(1, -1)

prediction = clf.predict(test_emb)

print("Prediction:", "Positive" if prediction[0] == 1 else "Negative")

Prediction: Negative
